# Hybrid Search for Agentic Systems: BM25 + kNN

> Based on: [How to Build Agentic RAG with Hybrid Search — Towards Data Science](https://towardsdatascience.com/how-to-build-agentic-rag-with-hybrid-search/)

---

## What We'll Cover

1. **Why pure vector search fails** — the vocabulary gap problem  
2. **BM25 sparse retrieval** — keyword matching with TF-IDF+ scoring  
3. **kNN dense retrieval** — semantic embeddings and HNSW  
4. **Hybrid fusion** — Reciprocal Rank Fusion (RRF) and weighted convex combination  
5. **Agentic RAG loop** — turning retrieval into an LLM-callable tool  
6. **Evaluation** — Hit Rate, MRR, NDCG@K  

---

### The Core Problem

A standard RAG pipeline looks like this:

```
User query → [Vector DB: kNN search] → Top-K chunks → LLM → Answer
```

This breaks when a user asks:
- `"Error code ER_NO_DEFAULT_FOR_FIELD"` — exact error string
- `"What does SKU-4821 cost?"` — product identifier
- `"Find the section about §3.2 compliance"` — legal clause reference

Vector embeddings **average meaning across dimensions** — rare exact identifiers get diluted.  
BM25 handles these perfectly. Hybrid search gets the best of both worlds.

## 1. Setup & Dependencies

In [1]:
%pip install -q rank_bm25 sentence-transformers faiss-cpu anthropic numpy scikit-learn python-dotenv --break-system-packages

Note: you may need to restart the kernel to use updated packages.


In [2]:
import math
import json
import numpy as np
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Any

# BM25
from rank_bm25 import BM25Okapi

# Dense embeddings + FAISS ANN index
from sentence_transformers import SentenceTransformer
import faiss

print("All imports OK")

/Users/danny.varod/GitHub/ronybot/Hybrid-Search-for-Agentic-Systems-Demo/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports OK


---
## 2. BM25 — Sparse Retrieval

### Theory

BM25 (*Best Match 25*) is the industry-standard keyword ranking algorithm. It scores every document $d$ against a query $q$ as:

$$\text{score}(d, q) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t,d) \cdot (k_1 + 1)}{f(t,d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}$$

| Symbol | Meaning |
|--------|---------|
| $f(t,d)$ | Term frequency of token $t$ in document $d$ |
| $\|d\|$ | Document length in tokens |
| avgdl | Average document length across corpus |
| $k_1$ | Term saturation (typically 1.2–2.0) |
| $b$ | Length normalization (typically 0.75) |
| IDF | $\log\frac{N - n_t + 0.5}{n_t + 0.5}$ — rewards rare terms |

**Key insight:** BM25 uses an **inverted index** (token → doc list). No GPU needed, sub-millisecond at scale.

### When BM25 wins
- Exact identifiers: error codes, SKUs, PINs, legal clause numbers  
- Rare technical terms that only appear in the target document  
- Queries where the user's vocabulary exactly matches the document's vocabulary

In [ ]:
# --- Sample corpus ---------------------------------------------------------
DOCS = [
    {"id": 0, "text": "Domestic bank transfers (ACH) normally complete within 1-2 business days."},
    {"id": 1, "text": "Wire transfers are usually processed on the same business day if submitted before the cutoff time."},
    {"id": 2, "text": "Checks generally take 2-5 business days to clear, depending on the issuing bank and amount."},
    {"id": 3, "text": "When a cheque is deposited, funds are placed on a temporary hold to ensure they can be collected from the paying bank."},
    {"id": 4, "text": "PENDING_KB200 means verification is still in progress."},
    {"id": 5, "text": "Status: COMPLETED indicates that funds from a transaction have successfully settled and are fully available."},
    {"id": 6, "text": "Status: RETURNED means the paying bank rejected the check, often due to insufficient funds (NSF)."},
    {"id": 7, "text": "For checks over $5,000, extended holds of up to 7 business days may apply per Regulation CC."},
    {"id": 8, "text": "New accounts (less than 30 days old) may face up to 9 business days of hold times on deposited checks."},
    {"id": 9, "text": "International checks can take significantly longer to clear, often 4-6 weeks."},
    {"id": 10, "text": "All checks must be properly endorsed on the back to avoid processing delays."},
    {"id": 11, "text": "A valid check must include the date, payee name, numeric amount, written amount, and signature of the payer."},
    {"id": 12, "text": "Mobile deposits might have additional processing time if the image is unclear or flagged for manual review."},
    {"id": 13, "text": "Customers can view their available balance versus current balance on the online banking portal."},
    {"id": 14, "text": "Current balance includes all transactions, including those on hold, whereas available balance only includes cleared funds."},
    {"id": 15, "text": "Status: PROCESSING indicates a wire or ACH transfer has been initiated but not yet concluded."},
    {"id": 16, "text": "To clear a check, the Federal Reserve Bank often routes the physical or digital image to the payer's financial institution."},
    {"id": 17, "text": "If a queue hold is extended, the bank must provide a written notice of delayed availability to the customer."},
    {"id": 18, "text": "Cash deposits are usually available immediately upon receipt by the teller."},
    {"id": 19, "text": "For wire transfers, the required fields include the recipient's routing number, account number, name, and address."},
    {"id": 20, "text": "A cashier's check is guaranteed by the bank and typically clears faster, often next-day availability."},
    {"id": 21, "text": "Money orders are similar to cashier's checks but usually have lower maximum limits."},
    {"id": 22, "text": "Status: PENDING_VERIFICATION is used when a transaction is flagged by anti-fraud systems."},
    {"id": 23, "text": "The routing number on a check identifies the specific financial institution it is drawn from."},
    {"id": 24, "text": "Checks with discrepancies between the numeric and written amounts will be processed based on the written amount."},
    {"id": 25, "text": "Post-dated checks cannot be deposited until the date written on the check arrives."},
    {"id": 26, "text": "Stale-dated checks, which are older than 6 months, may be refused by the bank."},
    {"id": 27, "text": "ACH transfers are settled in batches throughout the day rather than instantly."},
    {"id": 28, "text": "Status: CANCELLED means the payer or payee has revoked the transaction before processing began."},
    {"id": 29, "text": "Customer support should advise clients with cheques pending clearance to wait up to 5 business days for standard checks to clear."},
    {"id": 30, "text": "Status: PENDING_KB501 refers to ACH transfers that are currently queued for processing in the next batch window."},
    {"id": 31, "text": "Status: PENDING_KB105 means a cash deposit was made at an ATM but has not yet been physically verified by a teller."},
    {"id": 32, "text": "A transaction with PENDING_KB880 is awaiting manager approval due to exceeding typical transaction limits."},
    {"id": 33, "text": "The error code PENDING_KB009 indicates a mismatch in the routing number and requires manual customer verification."},
    {"id": 34, "text": "If a transfer is marked PENDING_KB422, it signifies a potential OFAC or AML hold pending further review."},
    {"id": 35, "text": "For international incoming wires, PENDING_KB991 indicates the funds are subject to currency conversion delays."},
    {"id": 36, "text": "PENDING_KB347 is a common code for mobile check deposits waiting for imaging clear validation."},
    {"id": 37, "text": "When an account goes into overdraft but has backup protection, transactions may briefly sit in PENDING_KB611."},
    {"id": 38, "text": "Status: PENDING_KB733 reflects an internal system delay syncing data between branches and the central mainframe."},
    {"id": 39, "text": "If a customer asks about PENDING_KB244, explain it means their replacement debit card issuance fee is processing."},
    {"id": 40, "text": "Please refer to article 200 in the compliance handbook for regulatory holds on large accounts."},
    {"id": 41, "text": "Our CRM limits the display of recent transaction history to 200 records per page for performance reasons."},
    {"id": 42, "text": "A standard personal check book contains 200 checks, but business accounts can order binders with 600 or more."},
    {"id": 43, "text": "There is a $200 minimum balance required to waive the monthly maintenance fee on basic savings accounts."},
    {"id": 44, "text": "ATM withdrawals are generally capped at $200 per day for new accounts until they pass their 30-day probationary period."},
    {"id": 45, "text": "The new terminal update requires a reboot; code PENDING_KB200 indicates the patch is stalled waiting for admin to check it."},
    {"id": 46, "text": "Status PENDING_KB200 check tray means the paper jam is cleared but the tray sensor has not yet reset."}
]

# Tokenise (whitespace + lowercase — use a proper tokeniser in production)
tokenised_corpus = [doc["text"].lower().split() for doc in DOCS]

bm25 = BM25Okapi(tokenised_corpus)

def bm25_search(query: str, top_k: int = 5) -> list[tuple[int, float]]:
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
    return [(doc_id, score) for doc_id, score in ranked[:top_k] if score > 0]

# Demo
input_query = "Why are the funds for a check still in PENDING_KB200 status?"
results = bm25_search(query=input_query, top_k=3)
print(f'BM25 results for: "{input_query}"')
for doc_id, score in results:
    print(f"  [{doc_id:02d}] score={score:.3f}  |  {DOCS[doc_id]['text']}")

BM25 results for: "Why are the funds for a check still in PENDING_KB200 status?"
  [04] score=10.053  |  PENDING_KB200 means verification is still in progress.
  [35] score=5.752  |  For international incoming wires, PENDING_KB991 indicates the funds are subject to currency conversion delays.
  [30] score=4.875  |  Status: PENDING_KB501 refers to ACH transfers that are currently queued for processing in the next batch window.


---
## 3. kNN Dense Retrieval

### Theory

Dense retrieval encodes both queries and documents into a **continuous vector space** using a transformer encoder (e.g. `all-MiniLM-L6-v2`, `text-embedding-3-small`).

At index time:
$$\mathbf{e}_d = \text{Encoder}(d) \in \mathbb{R}^{n}$$

At query time, find the $k$ nearest document vectors:
$$\text{sim}(\mathbf{e}_q, \mathbf{e}_d) = \frac{\mathbf{e}_q \cdot \mathbf{e}_d}{\|\mathbf{e}_q\| \cdot \|\mathbf{e}_d\|}$$

**HNSW (Hierarchical Navigable Small World)** is the standard ANN index — sub-linear lookup in $O(\log N)$ rather than brute-force $O(N)$.

### When dense retrieval wins
- Synonym bridging: `"slow database queries"` → finds `"PostgreSQL optimization techniques"`  
- Paraphrase matching: `"how to fix import errors"` → finds `"resolving module not found exceptions"`  
- Conversational, conceptual, or multi-hop questions

In [14]:
# Build a FAISS flat index (exact cosine via inner-product on normalised vectors)
model = SentenceTransformer("all-MiniLM-L6-v2")

corpus_texts = [doc["text"] for doc in DOCS]
corpus_embeddings = model.encode(corpus_texts, normalize_embeddings=True)  # L2-normalised → dot == cosine

dim = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # Inner Product on unit vectors = cosine similarity
index.add(corpus_embeddings.astype("float32"))

print(f"FAISS index built: {index.ntotal} vectors of dim={dim}")


def dense_search(query: str, top_k: int = 5) -> list[tuple[int, float]]:
    q_emb = model.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_emb, top_k)
    return [(int(idx), float(score)) for idx, score in zip(indices[0], scores[0]) if idx != -1]


# Demo — semantic query with no exact keyword overlap
input_query = "Why are the funds for a check still in PENDING_KB200 status?"
results = dense_search(query=input_query, top_k=3)
print(f'\nDense results for: "{input_query}"')
for doc_id, score in results:
    print(f"  [{doc_id:02d}] score={score:.3f}  |  {DOCS[doc_id]['text']}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12712.63it/s]


FAISS index built: 47 vectors of dim=384

Dense results for: "Why are the funds for a check still in PENDING_KB200 status?"
  [31] score=0.685  |  Status: PENDING_KB105 means a cash deposit was made at an ATM but has not yet been physically verified by a teller.
  [29] score=0.660  |  Customer support should advise clients with PENDING_KB200 to wait up to 5 business days for standard checks to clear.
  [04] score=0.657  |  PENDING_KB200 means verification is still in progress.


---
## 4. Fusion Strategies

### 4a. Reciprocal Rank Fusion (RRF)

RRF is **score-agnostic** — it ignores raw scores and only uses rank positions. This sidesteps the scale mismatch between BM25 (unbounded floats) and cosine similarity (−1 to 1).

$$\text{RRF}(d) = \sum_{r \in \text{retrievers}} \frac{1}{k + \text{rank}_r(d)}$$

- $k = 60$ is the constant from the original 2009 paper (empirically robust)
- A document ranked **#1** scores $\frac{1}{61} \approx 0.0164$
- A document appearing in **both** lists scores the sum of both contributions

**Properties:**
- Zero labelled data required
- Robust to score distribution mismatches
- Best starting point for any hybrid system

### 4b. Convex Combination (Weighted Score Fusion)

$$\text{score}(d) = \alpha \cdot \hat{s}_{\text{dense}}(d) + (1 - \alpha) \cdot \hat{s}_{\text{sparse}}(d)$$

Both scores are min-max normalised to $[0, 1]$ first. $\alpha$ controls the balance:

| $\alpha$ | Behaviour |
|----------|-----------|
| 1.0 | Pure dense (semantic) |
| 0.7 | Semantic-heavy — good for conversational queries |
| 0.5 | Balanced |
| 0.3 | Keyword-heavy — good for technical docs / error codes |
| 0.0 | Pure BM25 |

Requires 50–100 labelled query pairs to tune $\alpha$ effectively.

In [5]:
def reciprocal_rank_fusion(
    *ranked_lists: list[tuple[int, float]], k: int = 60
) -> list[tuple[int, float]]:
    """Merge ranked lists using RRF. Each list is [(doc_id, score), ...]."""
    rrf_scores: dict[int, float] = defaultdict(float)
    for ranked_list in ranked_lists:
        for rank, (doc_id, _score) in enumerate(ranked_list, start=1):
            rrf_scores[doc_id] += 1.0 / (k + rank)
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)


def minmax_normalize(scores: list[tuple[int, float]]) -> list[tuple[int, float]]:
    if not scores:
        return scores
    values = [s for _, s in scores]
    lo, hi = min(values), max(values)
    if hi == lo:
        return [(doc_id, 1.0) for doc_id, _ in scores]
    return [(doc_id, (s - lo) / (hi - lo)) for doc_id, s in scores]


def convex_combination(
    dense_results: list[tuple[int, float]],
    sparse_results: list[tuple[int, float]],
    alpha: float = 0.5,
) -> list[tuple[int, float]]:
    """Weighted score fusion. alpha=1 → pure dense; alpha=0 → pure BM25."""
    dense_norm = dict(minmax_normalize(dense_results))
    sparse_norm = dict(minmax_normalize(sparse_results))
    all_ids = set(dense_norm) | set(sparse_norm)
    combined = {
        doc_id: alpha * dense_norm.get(doc_id, 0.0) + (1 - alpha) * sparse_norm.get(doc_id, 0.0)
        for doc_id in all_ids
    }
    return sorted(combined.items(), key=lambda x: x[1], reverse=True)


def hybrid_search(
    query: str, top_k: int = 5, fusion: str = "rrf", alpha: float = 0.5
) -> list[tuple[int, float]]:
    """Run BM25 + kNN then fuse. fusion='rrf' or 'convex'."""
    bm25_results = bm25_search(query, top_k=top_k * 2)
    dense_results = dense_search(query, top_k=top_k * 2)

    if fusion == "rrf":
        merged = reciprocal_rank_fusion(bm25_results, dense_results)
    else:
        merged = convex_combination(dense_results, bm25_results, alpha=alpha)

    return merged[:top_k]


def print_results(label: str, results: list[tuple[int, float]]) -> None:
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print('─'*60)
    for doc_id, score in results:
        print(f"  [{doc_id:02d}] score={score:.4f}  |  {DOCS[doc_id]['text']}")

In [6]:
# ── Compare all three strategies ──────────────────────────────────────────

input_query = "Why are the funds for a check still in PENDING_KB200 status?"

print(f"\n{'═'*60}")
print(f'  QUERY: {input_query}')
print_results("BM25 only", bm25_search(query=input_query, top_k=3))
print_results("Dense (kNN) only", dense_search(query=input_query, top_k=3))
print_results("Hybrid — RRF (k=60)", hybrid_search(query=input_query, top_k=3, fusion="rrf"))
print_results("Hybrid — Convex α=0.5", hybrid_search(query=input_query, top_k=3, fusion="convex", alpha=0.5))


════════════════════════════════════════════════════════════
  QUERY: Why are the funds for a check still in PENDING_KB200 status?

────────────────────────────────────────────────────────────
  BM25 only
────────────────────────────────────────────────────────────
  [04] score=10.0636  |  PENDING_KB200 means verification is still in progress.
  [35] score=5.7615  |  For international incoming wires, PENDING_KB991 indicates the funds are subject to currency conversion delays.
  [03] score=5.5547  |  When a check is deposited, funds are placed on a temporary hold to ensure they can be collected from the paying bank.

────────────────────────────────────────────────────────────
  Dense (kNN) only
────────────────────────────────────────────────────────────
  [31] score=0.6846  |  Status: PENDING_KB105 means a cash deposit was made at an ATM but has not yet been physically verified by a teller.
  [29] score=0.6601  |  Customer support should advise clients with PENDING_KB200 to wait up t

---
## 5. Agentic RAG with Hybrid Search

### From passive pipeline to active agent

**Passive RAG** — retrieval happens once, before the LLM:
```
User → retrieve(query) → [chunks] → LLM → answer
```

**Agentic RAG** — the LLM *decides* when and how to retrieve:
```
User → LLM (with tools) ─┬─ retrieve(query_1) → evaluate → ...
                          ├─ retrieve(query_2, alpha=0.2) → ...
                          └─ synthesise → answer
```

Benefits:
1. **Query rewriting** — the LLM rephrases the query before retrieval for better recall  
2. **Iterative retrieval** — if the first batch isn't enough, the agent retrieves again  
3. **Dynamic alpha selection** — the agent picks sparse/dense weight based on query type  
4. **Multi-hop reasoning** — retrieve → reason → retrieve again → answer  

### Implementation with Claude's tool use API

In [7]:
import anthropic

# ── Tool definition exposed to Claude ───────────────────────────────────────
HYBRID_SEARCH_TOOL = {
    "name": "hybrid_search",
    "description": (
        "Search the document corpus using hybrid BM25 + kNN retrieval. "
        "Use alpha close to 0 for exact keyword queries (error codes, IDs), "
        "alpha close to 1 for semantic/conceptual queries, and alpha=0.5 for mixed queries."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query. Rephrase for better retrieval if needed.",
            },
            "top_k": {
                "type": "integer",
                "description": "Number of documents to return (default 3, max 5).",
                "default": 3,
            },
            "alpha": {
                "type": "number",
                "description": "Weight for dense (semantic) retrieval. 0=BM25 only, 1=kNN only, 0.5=balanced.",
                "default": 0.5,
            },
        },
        "required": ["query"],
    },
}


def execute_tool(tool_name: str, tool_input: dict) -> str:
    """Dispatch tool calls from the LLM to our local hybrid search."""
    if tool_name == "hybrid_search":
        query = tool_input["query"]
        top_k = min(int(tool_input.get("top_k", 3)), 5)
        alpha = float(tool_input.get("alpha", 0.5))
        results = hybrid_search(query, top_k=top_k, fusion="convex", alpha=alpha)
        hits = [
            {"doc_id": doc_id, "score": round(score, 4), "text": DOCS[doc_id]["text"]}
            for doc_id, score in results
        ]
        return json.dumps({"results": hits}, indent=2)
    return json.dumps({"error": f"Unknown tool: {tool_name}"})

In [8]:
def agentic_rag(user_question: str, max_iterations: int = 5, verbose: bool = True) -> str:
    """
    Agentic RAG loop.
    
    The agent autonomously decides:
    - What query to issue (query rewriting)
    - What alpha to use (sparse vs dense weight)
    - Whether to retrieve again (iterative retrieval)
    """
    client = anthropic.Anthropic()

    system_prompt = (
        "You are a helpful assistant with access to a document corpus via the hybrid_search tool. "
        "Before answering, always search for relevant documents. "
        "If the initial results are insufficient, search again with a different query or alpha value. "
        "Choose alpha carefully: use low alpha (≈0.1) for exact keyword lookups, "
        "high alpha (≈0.9) for conceptual questions, and 0.5 for general queries."
    )

    messages = [{"role": "user", "content": user_question}]
    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            system=system_prompt,
            tools=[HYBRID_SEARCH_TOOL],
            messages=messages,
        )

        if verbose:
            print(f"\n[Iteration {iteration}] stop_reason={response.stop_reason}")

        # Append assistant turn
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "end_turn":
            # Extract final text response
            for block in response.content:
                if hasattr(block, "text"):
                    return block.text
            return "(no text response)"

        if response.stop_reason == "tool_use":
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    if verbose:
                        print(f"  Tool call: {block.name}({json.dumps(block.input)})")
                    result = execute_tool(block.name, block.input)
                    if verbose:
                        print(f"  Tool result: {result[:300]}...")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    })
            messages.append({"role": "user", "content": tool_results})
        else:
            break

    return "(max iterations reached)"

### How the agentic loop works — step by step

The loop runs until the model either finishes (`end_turn`) or hits the `max_iterations` guard.

```
┌─────────────────────────────────────────────────────────────┐
│  User: "How can I make my database queries faster?"         │
└──────────────────────────┬──────────────────────────────────┘
                           │
            ┌──────────────▼──────────────┐
            │  Iteration 1                │
            │  Claude thinks: need data   │
            │  stop_reason = "tool_use"   │
            │  hybrid_search(             │
            │    query="db optimization", │
            │    alpha=0.9)               │
            └──────────────┬──────────────┘
                           │  ← tool result appended to messages
            ┌──────────────▼──────────────┐
            │  Iteration 2                │
            │  Claude thinks: also check  │
            │  indexing strategies        │
            │  stop_reason = "tool_use"   │
            │  hybrid_search(             │
            │    query="indexing slow",   │
            │    alpha=0.5)               │
            └──────────────┬──────────────┘
                           │  ← tool result appended to messages
            ┌──────────────▼──────────────┐
            │  Iteration 3                │
            │  Claude: enough context     │
            │  stop_reason = "end_turn"   │
            │  → returns final answer     │
            └─────────────────────────────┘
```

| `stop_reason`  | What happens next |
|----------------|------------------|
| `"tool_use"`   | Execute the tool, append result, **loop again** |
| `"end_turn"`   | Extract text from response, **return it** |
| anything else  | `break` — unexpected stop (e.g. `"max_tokens"`) |

In [9]:
# ── Simulated loop trace (no API key required) ──────────────────────────────
# This mirrors exactly what happens inside agentic_rag(), showing each
# iteration's stop_reason and what the loop does in response.

input_query = "Why are the funds for a check still in PENDING_KB200 status?"

SIMULATED_TURNS = [
    {
        "role": "user", 
        "content": input_query
    },
    {
        "stop_reason": "tool_use",
        "tool_name": "hybrid_search",
        "tool_input": {"query": "check holding times release conditions", "alpha": 0.8, "top_k": 3},
    },
    {
        "stop_reason": "tool_use",
        "tool_name": "hybrid_search",
        "tool_input": {"query": "PENDING_KB200 exact meaning", "alpha": 0.2, "top_k": 3},
    },
    {
        "stop_reason": "end_turn",
        "final_answer": (
            "Based on the retrieved documents, a check in PENDING_KB200 status means it is currently undergoing verification and funds aren't available yet (doc 4). Checks usually take 2-5 business days to clear (doc 2), but extended holds can apply depending on the check amount or account age (docs 7 & 8)."
        ),
    },
]

print(f'Question: "{input_query}"\n')
print("─" * 60)

messages = []  # grows with each iteration, just like the real loop

for i, turn in enumerate(SIMULATED_TURNS):
    if "role" in turn and turn["role"] == "user":
        messages.append(turn)
        continue

    print(f"\n[Iteration {i}] stop_reason = \"{turn['stop_reason']}\"")

    if turn["stop_reason"] == "tool_use":
        tool_input = turn["tool_input"]
        print(f"  → Claude calls: {turn['tool_name']}({json.dumps(tool_input)})")

        # Execute the real hybrid search
        result = execute_tool(turn["tool_name"], tool_input)
        hits = json.loads(result)["results"]

        print(f"  ← Tool returned {len(hits)} docs:")
        for h in hits:
            print(f"       [{h['doc_id']:02d}] score={h['score']:.4f}  |  {h['text']}")

        # Append tool result to messages — loop continues
        messages.append({"role": "tool_result", "content": result})
        print("  ↻  Appended result to messages — looping again")

    elif turn["stop_reason"] == "end_turn":
        print("  ✓  Model is satisfied — extracting final text and returning")
        print(f"\n── Final answer ──")
        print(turn["final_answer"])


Question: "Why are the funds for a check still in PENDING_KB200 status?"

────────────────────────────────────────────────────────────

[Iteration 1] stop_reason = "tool_use"
  → Claude calls: hybrid_search({"query": "check holding times release conditions", "alpha": 0.8, "top_k": 3})
  ← Tool returned 3 docs:
       [17] score=0.8000  |  If a check hold is extended, the bank must provide a written notice of delayed availability to the customer.
       [08] score=0.5241  |  New accounts (less than 30 days old) may face up to 9 business days of hold times on deposited checks.
       [25] score=0.1110  |  Post-dated checks cannot be deposited until the date written on the check arrives.
  ↻  Appended result to messages — looping again

[Iteration 2] stop_reason = "tool_use"
  → Claude calls: hybrid_search({"query": "PENDING_KB200 exact meaning", "alpha": 0.2, "top_k": 3})
  ← Tool returned 3 docs:
       [04] score=1.0000  |  PENDING_KB200 means verification is still in progress.
       

In [10]:
# ── Run the agent on two questions ──────────────────────────────────────────
# API key is loaded from a .env file in the same directory as this notebook.
# Create a .env file with: ANTHROPIC_API_KEY=sk-ant-...

import os
from dotenv import load_dotenv

load_dotenv()  # loads .env from the current working directory

if not os.environ.get("ANTHROPIC_API_KEY"):
    print("⚠  ANTHROPIC_API_KEY not set — skipping live API calls.\n"
          "Create a .env file with ANTHROPIC_API_KEY=sk-ant-... and re-run.")
else:
    print("=" * 60)
    print("Question 1: Exact keyword lookup")
    print("=" * 60)
    answer1 = agentic_rag("What does the status CANCELLED mean?")
    print("\n── Final answer ──")
    print(answer1)

    print("\n" + "=" * 60)
    print("Question 2: Semantic / conceptual query")
    print("=" * 60)
    answer2 = agentic_rag("Why are the funds for a check still in PENDING_KB200 status?")
    print("\n── Final answer ──")
    print(answer2)


Question 1: Exact keyword lookup

[Iteration 1] stop_reason=tool_use
  Tool call: hybrid_search({"query": "CANCELLED status meaning", "alpha": 0.3})
  Tool result: {
  "results": [
    {
      "doc_id": 28,
      "score": 1.0,
      "text": "Status: CANCELLED means the payer or payee has revoked the transaction before processing began."
    },
    {
      "doc_id": 5,
      "score": 0.0482,
      "text": "Status: COMPLETED indicates that funds from a transacti...
  Tool call: hybrid_search({"query": "status definitions and descriptions", "alpha": 0.7})
  Tool result: {
  "results": [
    {
      "doc_id": 5,
      "score": 1.0,
      "text": "Status: COMPLETED indicates that funds from a transaction have successfully settled and are fully available."
    },
    {
      "doc_id": 2,
      "score": 0.3,
      "text": "Checks generally take 2-5 business days to cle...

[Iteration 2] stop_reason=end_turn

── Final answer ──
The status **CANCELLED** means that the **payer or payee has revok

---
## 6. GraphRAG — Knowledge-Graph-Enhanced Retrieval

### What is GraphRAG?

Standard RAG treats documents as **isolated chunks**. GraphRAG enriches retrieval by first extracting a **knowledge graph** of entities and relationships, then using graph traversal to surface connected context that flat search would miss.

```
Documents → [Entity extraction] → Knowledge Graph
                                        │
Query → [Entity linking] → Seed nodes → [Graph traversal] → Neighborhood → LLM
```

Microsoft Research (2024) showed that GraphRAG substantially outperforms naive RAG on **global, multi-hop questions** that require synthesising information across many documents.

### Two retrieval modes

| Mode | How it works | Best for |
|------|-------------|---------|
| **Local search** | Expand the neighborhood of entities mentioned in the query | Specific entity lookups, relationship questions |
| **Global search** | Community summaries → map-reduce synthesis | "What are the main themes?" broad questions |

### Lightweight implementation

Below we build a minimal in-memory knowledge graph using only Python's standard library:

1. **Entity extraction** — pull noun phrases / identifiers from each document  
2. **Relationship extraction** — co-occurrence within the same document implies a relationship  
3. **Graph-augmented retrieval** — seed from hybrid search hits, then expand 1-hop neighbors

> In production you would use a dedicated graph DB (Neo4j, Kuzu) and an LLM for entity/relation extraction. This demo uses simple keyword heuristics to keep it dependency-free.


In [11]:
import re
from collections import defaultdict

# ── Step 1: Extract entities from each document ─────────────────────────────
# Simple heuristic: capitalized words, known identifiers, and domain terms.
ENTITY_PATTERNS = re.compile(
    r'\b([A-Z][A-Z0-9_\-]{2,}|BM25|kNN|HNSW|RRF|RAG|FAISS|'
    r'PostgreSQL|MySQL|Python|NDCG|MRR)\b'
)

def extract_entities(text: str) -> list[str]:
    return list({m.group(0) for m in ENTITY_PATTERNS.finditer(text)})

# Build entity → doc_ids index
entity_to_docs: dict[str, set[int]] = defaultdict(set)
doc_to_entities: dict[int, list[str]] = {}

for doc in DOCS:
    entities = extract_entities(doc["text"])
    doc_to_entities[doc["id"]] = entities
    for ent in entities:
        entity_to_docs[ent].add(doc["id"])

print("Knowledge graph stats:")
print(f"  Unique entities : {len(entity_to_docs)}")
print(f"  Entity → docs   : { {e: list(v) for e, v in list(entity_to_docs.items())[:8]} }")

# ── Step 2: Build doc-to-doc edges via shared entities ──────────────────────
doc_graph: dict[int, set[int]] = defaultdict(set)

for ent, doc_ids in entity_to_docs.items():
    doc_list = list(doc_ids)
    for i in range(len(doc_list)):
        for j in range(i + 1, len(doc_list)):
            doc_graph[doc_list[i]].add(doc_list[j])
            doc_graph[doc_list[j]].add(doc_list[i])

print(f"\nDocument graph edges (sample):")
for doc_id in list(doc_graph)[:4]:
    print(f"  doc {doc_id} → {sorted(doc_graph[doc_id])}")


# ── Step 3: Graph-augmented retrieval ───────────────────────────────────────
def graph_rag_search(
    query: str,
    top_k: int = 5,
    expand_hops: int = 1,
) -> list[tuple[int, float]]:
    """
    1. Seed with hybrid search results.
    2. Expand neighborhood up to `expand_hops` hops in the knowledge graph.
    3. Re-rank expanded set by dense similarity to the query.
    """
    # Seed retrieval
    seed_results = hybrid_search(query, top_k=top_k, fusion="rrf")
    seed_ids = {doc_id for doc_id, _ in seed_results}

    # Graph expansion
    expanded_ids = set(seed_ids)
    frontier = set(seed_ids)
    for _ in range(expand_hops):
        neighbors = set()
        for doc_id in frontier:
            neighbors |= doc_graph.get(doc_id, set())
        new_nodes = neighbors - expanded_ids
        expanded_ids |= new_nodes
        frontier = new_nodes

    # Re-rank expanded set by dense similarity
    q_emb = model.encode([query], normalize_embeddings=True).astype("float32")
    expanded_list = sorted(expanded_ids)
    exp_embs = corpus_embeddings[expanded_list]
    scores = (q_emb @ exp_embs.T).flatten()
    ranked = sorted(zip(expanded_list, scores.tolist()), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]


# ── Demo ────────────────────────────────────────────────────────────────────
print("\n" + "═" * 60)
print("  GraphRAG — Local search demo")
print("═" * 60)

for input_query in ["What does the status CANCELLED mean?", "Why are the funds for a check still in PENDING_KB200 status?"]:
    hybrid_res = hybrid_search(input_query, top_k=3, fusion="rrf")
    graph_res  = graph_rag_search(input_query, top_k=3, expand_hops=1)

    print(f"\nQuery: '{input_query}'")
    print(f"  Hybrid (seed) → docs: {[d for d,_ in hybrid_res]}")
    print(f"  Graph-expanded  → docs: {[d for d,_ in graph_res]}")
    print("  Graph-expanded results:")
    for doc_id, score in graph_res:
        print(f"    [{doc_id:02d}] score={score:.3f}  |  {DOCS[doc_id]['text']}")


Knowledge graph stats:
  Unique entities : 22
  Entity → docs   : {'ACH': [0, 27, 30, 15], 'PENDING_KB200': [45, 4, 29, 46], 'COMPLETED': [5], 'NSF': [6], 'RETURNED': [6], 'PROCESSING': [15], 'PENDING_VERIFICATION': [22], 'CANCELLED': [28]}

Document graph edges (sample):
  doc 0 → [15, 27, 30]
  doc 27 → [0, 15, 30]
  doc 30 → [0, 15, 27]
  doc 15 → [0, 27, 30]

════════════════════════════════════════════════════════════
  GraphRAG — Local search demo
════════════════════════════════════════════════════════════

Query: 'What does the status CANCELLED mean?'
  Hybrid (seed) → docs: [28, 16, 5]
  Graph-expanded  → docs: [28, 5, 16]
  Graph-expanded results:
    [28] score=0.802  |  Status: CANCELLED means the payer or payee has revoked the transaction before processing began.
    [05] score=0.495  |  Status: COMPLETED indicates that funds from a transaction have successfully settled and are fully available.
    [16] score=0.144  |  To clear a check, the Federal Reserve Bank often route

---
## 6. Evaluation Metrics

Three standard metrics for retrieval systems. All assume a **relevance judgement set** — a list of `(query, relevant_doc_ids)` pairs.

| Metric | What it measures | Formula |
|--------|-----------------|---------|
| **Hit Rate @K** | Does any relevant doc appear in top-K? | $\frac{1}{Q}\sum_q \mathbf{1}[\text{rel} \cap \text{top-K} \neq \emptyset]$ |
| **MRR** | How high is the *first* relevant doc? | $\frac{1}{Q}\sum_q \frac{1}{\text{rank}_{\text{first relevant}}}$ |
| **NDCG@K** | Graded relevance — rewards earlier hits more | $\frac{\text{DCG@K}}{\text{IDCG@K}}$ |

Higher is better for all three. NDCG@K is the most informative for production systems.

In [12]:
def hit_rate(results: list[tuple[int, float]], relevant_ids: set[int]) -> float:
    retrieved = {doc_id for doc_id, _ in results}
    return 1.0 if retrieved & relevant_ids else 0.0


def mrr(results: list[tuple[int, float]], relevant_ids: set[int]) -> float:
    for rank, (doc_id, _) in enumerate(results, start=1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(results: list[tuple[int, float]], relevant_ids: set[int], k: int) -> float:
    dcg = sum(
        1.0 / math.log2(rank + 1)
        for rank, (doc_id, _) in enumerate(results[:k], start=1)
        if doc_id in relevant_ids
    )
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(1, min(len(relevant_ids), k) + 1))
    return dcg / idcg if idcg > 0 else 0.0


# ── Mini evaluation on our small corpus ─────────────────────────────────────
EVAL_SET = [
    {"query": "Why are the funds for a check still in PENDING_KB200 status?", "relevant": {2, 4}},
    {"query": "What does the status CANCELLED mean?", "relevant": {28}},
    {"query": "Required fields for a check", "relevant": {11}},
    {"query": "How long do wire transfers take?", "relevant": {1}},
    {"query": "Difference between current and available balance", "relevant": {14}},
]

strategies = {
    "BM25 only":      lambda q: bm25_search(q, top_k=5),
    "Dense only":     lambda q: dense_search(q, top_k=5),
    "Hybrid RRF":     lambda q: hybrid_search(q, top_k=5, fusion="rrf"),
    "Hybrid α=0.5":   lambda q: hybrid_search(q, top_k=5, fusion="convex", alpha=0.5),
    "GraphRAG":       lambda q: graph_rag_search(q, top_k=5, expand_hops=1),
}

print(f"{'Strategy':<20} {'Hit@3':>8} {'MRR':>8} {'NDCG@3':>8}")
print("─" * 48)

for name, retriever in strategies.items():
    hits, mrrs, ndcgs = [], [], []
    for item in EVAL_SET:
        res = retriever(item["query"])
        rel = item["relevant"]
        hits.append(hit_rate(res[:3], rel))
        mrrs.append(mrr(res, rel))
        ndcgs.append(ndcg_at_k(res, rel, k=3))
    print(f"{name:<20} {np.mean(hits):>8.3f} {np.mean(mrrs):>8.3f} {np.mean(ndcgs):>8.3f}")


Strategy                Hit@3      MRR   NDCG@3
────────────────────────────────────────────────
BM25 only               0.800    0.700    0.649
Dense only              1.000    0.867    0.861
Hybrid RRF              1.000    0.800    0.775
Hybrid α=0.5            1.000    0.900    0.849
GraphRAG                1.000    0.900    0.877


---
## 7. Decision Guide — Which Strategy to Use?

```
                     ┌─────────────────────────────────────┐
                     │        What kind of query?           │
                     └──────────────┬──────────────────────┘
                                    │
              ┌─────────────────────┼─────────────────────┐
              ▼                     ▼                     ▼
        Exact identifiers    Mixed / general        Conceptual /
        (error codes,        queries                semantic
         SKUs, IDs)                                 (synonyms,
              │                     │                paraphrase)
              ▼                     ▼                     ▼
       Convex α ≈ 0.1        RRF (k=60)           Convex α ≈ 0.9
       or BM25 only          or Convex α=0.5      or Dense only
```

| Situation | Fusion | Alpha |
|-----------|--------|-------|
| Mixed queries, no labelled data | **RRF** k=60 | — |
| 50+ labelled pairs available | **Convex** | Tune via grid search |
| Technical docs, error codes, APIs | Convex | 0.2–0.3 |
| Conversational, support KB | Convex | 0.7–0.8 |
| Agentic system | Let the **LLM choose** alpha per query | Dynamic |

> **Empirical gains on BEIR benchmark:**  
> Hybrid vs dense-only: **+26–31% NDCG** on domains with high vocabulary mismatch  
> Hybrid vs BM25-only: **+18.5% MRR** on semantic / paraphrase-heavy domains

---
## 8. Production Checklist

```
□  Tokenisation        Use a proper tokeniser (NLTK, spaCy) — not just whitespace split
□  Stopword removal    Strip "the/a/is" before BM25 indexing
□  Chunking strategy   500–1000 token chunks with 10–20% overlap
□  Embedding model     Match embedding model to your domain (fine-tune if needed)
□  Index type          FAISS HNSW (local) / Qdrant / Weaviate / Elasticsearch (production)
□  Reranking           Add a cross-encoder reranker (e.g. ms-marco-MiniLM) on top-20 hits
□  Evaluation          Build a ≥100 query evaluation set before tuning alpha
□  Caching             Cache embeddings at index time; cache query embeddings for repeat queries
□  Monitoring          Log alpha values the agent picks — drift signals corpus change
```

## Sources

- [How to Build Agentic RAG with Hybrid Search — Towards Data Science](https://towardsdatascience.com/how-to-build-agentic-rag-with-hybrid-search/)
- [Hybrid Search for RAG: BM25, SPLADE, and Vector Search Combined — PremAI Blog](https://blog.premai.io/hybrid-search-for-rag-bm25-splade-and-vector-search-combined/)
- [Hybrid Search Explained — Weaviate](https://weaviate.io/blog/hybrid-search-explained)
- [Introducing Reciprocal Rank Fusion — OpenSearch](https://opensearch.org/blog/introducing-reciprocal-rank-fusion-hybrid-search/)
- [Full-text search for RAG apps: BM25 & hybrid search — Redis](https://redis.io/blog/full-text-search-for-rag-the-precision-layer/)